# Data Audit

In [2]:
import pandas as pd
import numpy as np

 ## Load Datasets

In [3]:
# Load all CSV files from the raw data directory into a dictionary of DataFrames
from pathlib import Path

data_path = Path("../data/raw")
csv_files = list(data_path.glob("*.csv"))

datasets = {}
for file in csv_files:
    datasets[file.stem] = pd.read_csv(file)

## Dataset Shape & Overview

In [4]:
# Summarize each dataset: shape, duplicates, missing values, memory usage, and dtypes
def audit_datasets(datasets):
    rows = []
    for name, df in datasets.items():
        n_rows, n_cols = df.shape
        n_dup_rows = df.duplicated().sum()
        n_unique_rows = df.drop_duplicates().shape[0]
        total_cells = n_rows * n_cols
        missing_cells = df.isna().sum().sum()
        
        rows.append({
            'file': name,
            'rows': n_rows,
            'cols': n_cols,
            'unique_rows': n_unique_rows,
            'dup_rows': n_dup_rows,
            'dup_%': round(n_dup_rows / n_rows * 100, 2) if n_rows else 0,
            'missing_cells': missing_cells,
            'missing_%': round(missing_cells / total_cells * 100, 2) if total_cells else 0,
            'memory_MB': round(df.memory_usage(deep=True).sum() / 1024**2, 2),
            'dtypes': df.dtypes.astype(str).value_counts().to_dict(),
        })
    
    return pd.DataFrame(rows).sort_values('rows', ascending=False).reset_index(drop=True)

# Run the audit and display results sorted by row count
audit_datasets(datasets)

,file,rows,cols,unique_rows,dup_rows,dup_%,missing_cells,missing_%,memory_MB,dtypes
0,olist_geolocation_dataset,1000163,5,738332,261831,26.18,0,0.00,146.09,"{'float64': 2, 'object': 2, 'int64': 1}"
1,olist_order_items_dataset,112650,7,112650,0,0.00,0,0.00,39.43,"{'object': 4, 'float64': 2, 'int64': 1}"
2,olist_order_payments_dataset,103886,5,103886,0,0.00,0,0.00,17.81,"{'object': 2, 'int64': 2, 'float64': 1}"
3,olist_customers_dataset,99441,5,99441,0,0.00,0,0.00,29.62,"{'object': 4, 'int64': 1}"
4,olist_orders_dataset,99441,8,99441,0,0.00,4908,0.62,58.97,{'object': 8}
5,olist_order_reviews_dataset,99224,7,99224,0,0.00,145903,21.01,42.75,"{'object': 6, 'int64': 1}"
6,olist_products_dataset,32951,9,32951,0,0.00,2448,0.83,6.79,"{'float64': 7, 'object': 2}"
7,olist_sellers_dataset,3095,4,3095,0,0.00,0,0.00,0.66,"{'object': 3, 'int64': 1}"
8,product_category_name_translation,71,2,71,0,0.00,0,0.00,0.01,{'object': 2}


 ## Column-Level Audit

In [5]:
# Flatten all columns across datasets into a single table with dtype and missing-rate info
col_detail = pd.DataFrame([
    {'file': name, 'position': i, 'column': col, 'dtype': str(df[col].dtype), 'missing value': round(df[col].isnull().mean() * 100, 3)}
    for name, df in datasets.items()
    for i, col in enumerate(df.columns)
])

# Run the audit and display results sorted by row count
col_detail

,file,position,column,dtype,missing value
0,olist_geolocation_dataset,0,geolocation_zip_code_prefix,int64,0.000
1,olist_geolocation_dataset,1,geolocation_lat,float64,0.000
2,olist_geolocation_dataset,2,geolocation_lng,float64,0.000
3,olist_geolocation_dataset,3,geolocation_city,object,0.000
4,olist_geolocation_dataset,4,geolocation_state,object,0.000
5,olist_order_payments_dataset,0,order_id,object,0.000
6,olist_order_payments_dataset,1,payment_sequential,int64,0.000
7,olist_order_payments_dataset,2,payment_type,object,0.000
8,olist_order_payments_dataset,3,payment_installments,int64,0.000
9,olist_order_payments_dataset,4,payment_value,float64,0.000


 ## Candidate Keys

In [6]:
# Map each dataset to its ID-like columns for key candidacy testing
id_columns = {
    "olist_orders_dataset": ["order_id", "customer_id"],
    "olist_customers_dataset": ["customer_id", "customer_unique_id"],
    "olist_order_items_dataset": ["order_id", "order_item_id", "product_id", "seller_id"],
    "olist_order_reviews_dataset": ["review_id", "order_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "olist_geolocation_dataset": ["geolocation_zip_code_prefix"],
    "product_category_name_translation": ["product_category_name"],
}

# Check single-column uniqueness: flag columns that could serve as primary keys
def analyze_duplicates(datasets, id_columns):
    rows = []
    for name, df in datasets.items():
        for col in id_columns.get(name, []):
            if col in df.columns:
                rows.append({
                    'file': name,
                    'column': col,
                    'unique': df[col].nunique(),
                    'rows': len(df),
                    'duplicated': df[col].duplicated().sum(),
                    'candidate key': '🟢' if df[col].duplicated().sum() == 0 and df[col].nunique() ==len(df) else '🔴'
                })
    return pd.DataFrame(rows)

# Run single-column key analysis
composite_key = analyze_duplicates(datasets, id_columns)
composite_key

,file,column,unique,rows,duplicated,candidate key
0,olist_geolocation_dataset,geolocation_zip_code_prefix,19015,1000163,981148,🔴
1,olist_order_payments_dataset,order_id,99440,103886,4446,🔴
2,olist_order_payments_dataset,payment_sequential,29,103886,103857,🔴
3,olist_sellers_dataset,seller_id,3095,3095,0,🟢
4,olist_order_items_dataset,order_id,98666,112650,13984,🔴
5,olist_order_items_dataset,order_item_id,21,112650,112629,🔴
6,olist_order_items_dataset,product_id,32951,112650,79699,🔴
7,olist_order_items_dataset,seller_id,3095,112650,109555,🔴
8,olist_order_reviews_dataset,review_id,98410,99224,814,🔴
9,olist_order_reviews_dataset,order_id,98673,99224,551,🔴


In [7]:
# Define composite key candidates (multi-column combinations) for key testing
id_composite_columns = {
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_reviews_dataset": ["review_id", "order_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
}

# Check whether each column combination uniquely identifies rows (no duplicates, no nulls)
def analyze_duplicates(datasets, id_columns):
    rows = []
    for name, df in datasets.items():
        cols = id_columns.get(name, [])
        cols = [c for c in cols if c in df.columns]
        if not cols:
            continue
        rows.append({
            'file': name,
            'columns': ", ".join(cols),
            'unique': df[cols].drop_duplicates().shape[0],
            'rows': len(df),
            'duplicated': df.duplicated(subset=cols).sum(),
            'nulls': int(df[cols].isna().sum().sum()),
            'candidate key': '🟢' if (df.duplicated(subset=cols).sum() == 0 
                                       and df[cols].isna().sum().sum() == 0) 
                              else '🔴'
        })
    return pd.DataFrame(rows)

# Run composite key analysis
composite_key = analyze_duplicates(datasets, id_composite_columns)
composite_key

,file,columns,unique,rows,duplicated,nulls,candidate key
0,olist_order_payments_dataset,"order_id, payment_sequential",103886,103886,0,0,🟢
1,olist_order_items_dataset,"order_id, order_item_id",112650,112650,0,0,🟢
2,olist_order_reviews_dataset,"review_id, order_id",99224,99224,0,0,🟢


## Table Relationships Summary

| Table | Grain (one row represents...) | Rows | Candidate Key |
|---|---|---:|---|
| `orders` | One order | 99,441 | `order_id` |
| `customers` | One customer record | 99,441 | `customer_id` |
| `order_items` | One item within an order | 112,650 | `order_id + order_item_id` |
| `payments` | One payment record within an order | 103,886 | `order_id + payment_sequential` |
| `reviews` | One review record associated with an order | 99,224 | `review_id + order_id` |
| `products` | One product | 32,951 | `product_id` |
| `sellers` | One seller | 3,095 | `seller_id` |
| `geolocation` | One geolocation record | 1,000,163 | — |
| `translation` | One product category and its English translation | 71 | `product_category_name` |

| Parent Table | Parent Key | Child Table | Foreign Key | Relationship |
|---|---|---|---|---|
| `customers` | `customer_id` | `orders` | `customer_id` | 1:N |
| `orders` | `order_id` | `order_items` | `order_id` | 1:N |
| `products` | `product_id` | `order_items` | `product_id` | 1:N |
| `sellers` | `seller_id` | `order_items` | `seller_id` | 1:N |
| `orders` | `order_id` | `payments` | `order_id` | 1:N |
| `orders` | `order_id` | `reviews` | `order_id` | 1:N* |
| `translation` | `product_category_name` | `products` | `product_category_name` | 1:N |

> *The `orders` → `reviews` relationship requires further investigation because `order_id` is not unique in the reviews table.

In [12]:
# Define relationships to validate: (child_table, child_col, parent_table, parent_col)
relationships = [
    ("olist_orders_dataset",         "customer_id", "olist_customers_dataset",  "customer_id"),
    ("olist_order_items_dataset",    "product_id",  "olist_products_dataset",   "product_id"),
    ("olist_order_items_dataset",    "seller_id",   "olist_sellers_dataset",    "seller_id"),
    ("olist_order_items_dataset",    "order_id",    "olist_orders_dataset",     "order_id"),
    ("olist_order_payments_dataset", "order_id",    "olist_orders_dataset",     "order_id"),
    ("olist_order_reviews_dataset",  "order_id",    "olist_orders_dataset",     "order_id"),
]

def validate_relationships(datasets, relationships):
    rows = []
    for child, child_col, parent, parent_col in relationships:
        if child not in datasets or parent not in datasets:
            continue
        c, p = datasets[child], datasets[parent]
        if child_col not in c.columns or parent_col not in p.columns:
            continue
        matched   = c[child_col].isin(p[parent_col]).sum()
        unmatched = len(c) - matched
        rows.append({
            "from": f"{child}.{child_col}",
            "to":   f"{parent}.{parent_col}",
            "matched": matched,
            "unmatched": unmatched,
            "match_%": round(matched / len(c) * 100, 2) if len(c) else 0,
            "valid": "🟢" if unmatched == 0 else "🔴",
        })
    return pd.DataFrame(rows)

validate_relationships(datasets, relationships)

,from,to,matched,unmatched,match_%,valid
0,olist_orders_dataset.customer_id,olist_customers_dataset.customer_id,99441,0,100.0,🟢
1,olist_order_items_dataset.product_id,olist_products_dataset.product_id,112650,0,100.0,🟢
2,olist_order_items_dataset.seller_id,olist_sellers_dataset.seller_id,112650,0,100.0,🟢
3,olist_order_items_dataset.order_id,olist_orders_dataset.order_id,112650,0,100.0,🟢
4,olist_order_payments_dataset.order_id,olist_orders_dataset.order_id,103886,0,100.0,🟢
5,olist_order_reviews_dataset.order_id,olist_orders_dataset.order_id,99224,0,100.0,🟢


In [8]:
reviews = datasets["olist_order_reviews_dataset"]

duplicate_review_ids = (
    reviews[reviews["review_id"].duplicated(keep=False)]
    .sort_values("review_id")
)

duplicate_review_ids

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
...,...,...,...,...,...,...,...
31120,fe5c833752953fed3209646f1f63b53c,4863e15fa53273cc7219c58f5ffda4fb,1,NaN,"Comprei dois produtos e ambos, mesmo enviados ...",2018-02-28 00:00:00,2018-02-28 13:57:52
7870,ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
73951,ffb8cff872a625632ac983eb1f88843c,c44883fc2529b4aa03ca90e7e09d95b6,3,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:41:07


In [9]:
duplicate_review_ids.groupby("review_id")["order_id"].nunique().value_counts()

order_id
2    764
3     25
Name: count, dtype: int64

In [10]:
duplicate_review_ids.groupby("review_id").size().value_counts()

2    764
3     25
Name: count, dtype: int64

In [13]:
reviews.groupby("order_id")["review_score"].nunique().value_counts()

review_score
1    98471
2      202
Name: count, dtype: int64

In [14]:
multi_score_orders = reviews.groupby("order_id")["review_score"].nunique()
multi_score_orders[multi_score_orders > 1].count()

np.int64(202)

In [ ]:
review_group = reviews.groupby("order_id").agg(
    review_count=("review_id", "count"),
    score_count=("review_score", "nunique"),
    message_count=("review_comment_message", "nunique")
).query("review_count > 1").head(20)

review_group

,review_count,score_count,message_count
order_id,,,
0035246a40f520710769010f752e7507,2,1,1
013056cfe49763c6f66bda03396c5ee3,2,2,0
0176a6846bcb3b0d3aa3116a9a768597,2,1,0
02355020fd0a40a0d56df9f6ff060413,2,2,2
029863af4b968de1e5d6a82782e662f5,2,2,0
02e0b68852217f5715fb9cc885829454,2,1,2
02e723e8edb4a123d414f56cc9c4665e,2,1,1
03515a836bb855b03f7df9dee520a8fc,2,1,0
03c939fd7fd3b38f8485a0f95798f1f6,3,2,2


In [29]:
review_group = (
    reviews
    .groupby("order_id")
    .agg(
        review_count=("review_id", "count"),
        score_count=("review_score", "nunique"),
        message_present_count=("review_comment_message", "count"),
        message_unique_count=("review_comment_message", "nunique")
    )
    .reset_index()
)

In [30]:
def classify_reviews(row):
    if row["review_count"] == 1:
        return "single"
    if row["score_count"] > 1:
        return "conflicting_scores"
    if row["message_present_count"] == 0:
        return "same_score_no_message"
    if row["message_unique_count"] == 1:
        return "same_score_same_message"
    return "same_score_different_message"

review_group["review_pattern"] = review_group.apply(classify_reviews, axis=1)

In [31]:
pattern_summary = (
    review_group["review_pattern"]
    .value_counts()
    .rename_axis("review_pattern")
    .reset_index(name="orders")
)

pattern_summary["percentage"] = (
    pattern_summary["orders"] / pattern_summary["orders"].sum() * 100
).round(2)

pattern_summary

,review_pattern,orders,percentage
0,single,98126,99.45
1,same_score_no_message,216,0.22
2,conflicting_scores,202,0.20
3,same_score_different_message,77,0.08
4,same_score_same_message,52,0.05


In [32]:
for pattern in ["same_score_no_message", 
                "same_score_same_message",
                "same_score_different_message",
                "conflicting_scores"]:
    
    print(f"\n{'='*70}\n{pattern}\n{'='*70}")
    
    order_ids = review_group[
        review_group["review_pattern"] == pattern
    ]["order_id"].head(3)
    
    sample = reviews[reviews["order_id"].isin(order_ids)].sort_values("order_id")
    print(sample[["order_id", "review_id", "review_score", 
                  "review_comment_message"]].to_string())


same_score_no_message
                               order_id                         review_id  review_score review_comment_message
854    0176a6846bcb3b0d3aa3116a9a768597  830636803620cdf8b6ffaf1b2f6e92b2             5                    NaN
83224  0176a6846bcb3b0d3aa3116a9a768597  d8e8c42271c8fb67b9dad95d98c8ff80             5                    NaN
38386  03515a836bb855b03f7df9dee520a8fc  1dee9d450a2b8533d2085cef898ca4a2             5                    NaN
49625  03515a836bb855b03f7df9dee520a8fc  5e1ee2e924a1dd10d80b99555383c0a7             5                    NaN
33045  05452881e2846549b81d39249bb66ad7  61bb221369ffa0c6b0df900f97c5062b             5                    NaN
47676  05452881e2846549b81d39249bb66ad7  5710a8cfcb822bc6f62ece1753b11786             5                    NaN

same_score_same_message
                               order_id                         review_id  review_score                                                                                        